In [2]:
import pandas as pd

In [3]:
original_data = pd.read_csv("uk_churches.csv")

In [17]:
original_data

,charity_name,registered_number,contact_postcode,contact_web,income,expenditure,denomination,principal_bank,reserves,assets_total,assets_cash,funds_total
0,HITCHAM FREE CHURCH,200027,NaN,NaN,93073.0,264019.0,Non‑denominational,Caf Bank,2024499.0,2077693.0,30731.0,2838807.0
1,MELLS CHURCH LANDS,200999,NaN,NaN,10232.0,5511.0,Church of England,NaN,NaN,NaN,NaN,NaN
2,CHURCH AND TOWN ESTATE (EXCLUDING CHURCH),201239,NaN,NaN,NaN,NaN,Church of England,NaN,NaN,NaN,NaN,NaN
3,CHURCH AND POOR LANDS FOR PARISH OF ST JOHN,201444,NaN,NaN,29224.0,13918.0,Church of England,Lloyds,NaN,NaN,NaN,NaN
4,CHURCH AND POORS ESTATE,201560,NaN,NaN,NaN,NaN,Church of England,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
31686,BRIDGE CHURCH FOR LIFE,1218016,BS32 0DA,NaN,NaN,NaN,Unknown,NaN,NaN,NaN,NaN,NaN
31687,NEW COVENANT CHURCH TRANSFORMING LIVES CITY,1218032,SE28 8QE,NaN,NaN,NaN,Unknown,NaN,NaN,NaN,NaN,NaN
31688,THE PAROCHIAL CHURCH COUNCIL OF THE ECCLESIAST...,1218085,WS14 9LE,NaN,NaN,NaN,Unknown,NaN,NaN,NaN,NaN,NaN
31689,IGLESIA EVANGELICA VIDA Y GRACIA LONDON,1218170,CR4 3AR,NaN,NaN,NaN,Evangelical,NaN,NaN,NaN,NaN,NaN


In [23]:
categories_orig = \
    (
        pd
        .read_json(
            "publicextract.charity_classification\\publicextract.charity_classification.json",
            encoding="utf-8-sig",
        )
    )

In [24]:
categories=\
    (
    categories_orig
    .query("classification_code==108")
    [["registered_charity_number"]]
    
)

In [25]:
category_numbers = (
    categories.loc[
        categories["registered_charity_number"].notna(),
        ["registered_charity_number"],
    ]
    .drop_duplicates()
    .assign(category_match="Y")
)

In [26]:
category_numbers

,registered_charity_number,category_match
40,200023,Y
50,200027,Y
78,200036,Y
89,200041,Y
182,200059,Y
...,...,...
1748064,1219795,Y
1748072,1219921,Y
1748079,1219970,Y
1748117,1219944,Y


In [27]:
original_data = original_data.merge(
    category_numbers,
    left_on="registered_number",
    right_on="registered_charity_number",
    how="left",
    validate="many_to_one",
).drop(columns="registered_charity_number")

original_data

,charity_name,registered_number,contact_postcode,contact_web,income,expenditure,denomination,principal_bank,reserves,assets_total,assets_cash,funds_total,category_match_x,category_match_y
0,HITCHAM FREE CHURCH,200027,NaN,NaN,93073.0,264019.0,Non‑denominational,Caf Bank,2024499.0,2077693.0,30731.0,2838807.0,Y,Y
1,MELLS CHURCH LANDS,200999,NaN,NaN,10232.0,5511.0,Church of England,NaN,NaN,NaN,NaN,NaN,Y,NaN
2,CHURCH AND TOWN ESTATE (EXCLUDING CHURCH),201239,NaN,NaN,NaN,NaN,Church of England,NaN,NaN,NaN,NaN,NaN,Y,NaN
3,CHURCH AND POOR LANDS FOR PARISH OF ST JOHN,201444,NaN,NaN,29224.0,13918.0,Church of England,Lloyds,NaN,NaN,NaN,NaN,Y,Y
4,CHURCH AND POORS ESTATE,201560,NaN,NaN,NaN,NaN,Church of England,NaN,NaN,NaN,NaN,NaN,Y,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
31686,BRIDGE CHURCH FOR LIFE,1218016,BS32 0DA,NaN,NaN,NaN,Unknown,NaN,NaN,NaN,NaN,NaN,Y,Y
31687,NEW COVENANT CHURCH TRANSFORMING LIVES CITY,1218032,SE28 8QE,NaN,NaN,NaN,Unknown,NaN,NaN,NaN,NaN,NaN,Y,Y
31688,THE PAROCHIAL CHURCH COUNCIL OF THE ECCLESIAST...,1218085,WS14 9LE,NaN,NaN,NaN,Unknown,NaN,NaN,NaN,NaN,NaN,Y,Y
31689,IGLESIA EVANGELICA VIDA Y GRACIA LONDON,1218170,CR4 3AR,NaN,NaN,NaN,Evangelical,NaN,NaN,NaN,NaN,NaN,Y,Y


In [28]:
(
    original_data
    .to_csv(
        "uk_churches_trimmed.csv",
        index=False,
        encoding="utf-8-sig",
    )
)